# Stage 3 — Full Pipeline (Stage 1 segmentation + Stage 2 disease classification)

This notebook imports the **Stage 1** Mask R-CNN and **Stage 2** ResNet-34 classifier
as Kaggle dataset inputs (no retraining happens here) and runs them together on a
panoramic X-ray:

1. Stage 1 detects every tooth, crops to the teeth region, colors each tooth by
   instance, splits into quadrants (UL/UR/LL/LR), and numbers each tooth 1-8 from
   the midline outward — **identical to the Stage 1 notebook's own visualization**,
   reused verbatim below.
2. For every detected tooth, Stage 2 classifies it as Healthy or one of 4 disease
   classes from a padded crop of the same tooth.
3. A 6th panel renders the same panoramic X-ray colored by **disease status** instead
   of by instance — Healthy teeth in green, each disease class in its own color —
   with quadrant + tooth number + status labeled on every tooth, plus a clinical
   report (text) for the whole image.

**Running this against the current Stage 2 checkpoint** (the one where Periapical
Lesion collapsed to 0 recall) is intentional for now — this notebook is meant to
validate the *pipeline plumbing* end-to-end. Don't be surprised if Periapical Lesion
never appears in the disease-colored panel below; that's the known limitation
carrying over from Stage 2, not a new bug here.

## Cell map

| Cell | Purpose |
|------|---------|
| 1 | Setup: imports, paths (Stage 1 + Stage 2 weights, test images), config |
| 2 | Load Stage 1 Mask R-CNN |
| 3 | Load Stage 2 ResNet-34 classifier + checkpoint metadata |
| 4 | Stage 1 inference + quadrant/numbering helpers (ported verbatim) |
| 5 | Per-tooth Stage 2 classification helper |
| 6 | Full pipeline visualization + clinical report function |
| 7 | Run on sample test images |
| 8 | Save pipeline manifest + sample outputs |


In [ ]:
# ================== CELL 1: SETUP ==================
import os, json, random, time
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms.functional as TF
from torchvision import transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import nms

print('='*70)
print('[CELL 1] STAGE 3 — FULL PIPELINE SETUP')
print('='*70)
print(f'  PyTorch:     {torch.__version__}')
print(f'  Torchvision: {torchvision.__version__}')
print(f'  CUDA:        {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU:         {torch.cuda.get_device_name(0)}')

def first_existing(candidates):
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

# ── Stage 1 weights ─────────────────────────────────────────────────────────
STAGE1_WEIGHTS_CANDIDATES = [
    '/kaggle/input/datasets/ethelrani/maskrcnn-teeth-stage1-weights/maskrcnn_teeth_best.pth',
    '/kaggle/input/maskrcnn-teeth-stage1-weights/maskrcnn_teeth_best.pth',
]
STAGE1_WEIGHTS = first_existing(STAGE1_WEIGHTS_CANDIDATES)

# ── Stage 2 weights ──────────────────────────────────────────────────────────
# Create a Kaggle dataset from the Stage 2 notebook's /kaggle/working output
# (stage2_disease_best.pth) and attach it here, same as you did for Stage 1.
STAGE2_WEIGHTS_CANDIDATES = [
    '/kaggle/input/datasets/ethelrani/stage2-disease-classifier-weights/stage2_disease_best.pth',
    '/kaggle/input/stage2-disease-classifier-weights/stage2_disease_best.pth',
    '/kaggle/input/datasets/ethelrani/dental-stage2-disease-classifier/stage2_disease_best.pth',
]
STAGE2_WEIGHTS = first_existing(STAGE2_WEIGHTS_CANDIDATES)

# ── Test images: DENTEX validation X-rays (unseen, blind test) ─────────────
DENTEX_ROOT_CANDIDATES = [
    '/kaggle/input/datasets/truthisneverlinear/dentex-challenge-2023',
    '/kaggle/input/dentex-challenge-2023',
]
DENTEX_ROOT = first_existing(DENTEX_ROOT_CANDIDATES)

_test_img_dir_candidates = []
if DENTEX_ROOT:
    _test_img_dir_candidates = [
        os.path.join(DENTEX_ROOT, 'validation_data/validation_data/quadrant_enumeration_disease/xrays'),
        os.path.join(DENTEX_ROOT, 'validation_data/validation_data/quadrant-enumeration-disease/xrays'),
        os.path.join(DENTEX_ROOT, 'validation_data/quadrant_enumeration_disease/xrays'),
    ]
TEST_IMG_DIR = first_existing(_test_img_dir_candidates)

_fallback_train_img_dir = None
if DENTEX_ROOT:
    _fallback_train_img_dir = first_existing([
        os.path.join(DENTEX_ROOT, 'training_data/training_data/quadrant-enumeration-disease/xrays'),
        os.path.join(DENTEX_ROOT, 'training_data/quadrant-enumeration-disease/xrays'),
    ])

print()
print('[CELL 1] Path discovery:')
for label, path in [
    ('Stage1 weights      ', STAGE1_WEIGHTS),
    ('Stage2 weights      ', STAGE2_WEIGHTS),
    ('DENTEX root         ', DENTEX_ROOT),
    ('Test images (val)   ', TEST_IMG_DIR),
    ('Fallback train xrays', _fallback_train_img_dir),
]:
    status = '✓' if (path and os.path.exists(path)) else '✗ MISSING'
    print(f'  {status}  {label}: {path}')

if STAGE1_WEIGHTS is None:
    raise FileNotFoundError('❌ Stage 1 weights not found. Attach the Stage 1 weights dataset.')
if STAGE2_WEIGHTS is None:
    raise FileNotFoundError(
        '❌ Stage 2 weights not found. Create a Kaggle dataset from the '
        "Stage 2 notebook's /kaggle/working/stage2_disease_best.pth output and attach it here, "
        'then add its real path to STAGE2_WEIGHTS_CANDIDATES above if it differs.'
    )
if TEST_IMG_DIR is None:
    if _fallback_train_img_dir:
        print()
        print('  ⚠  No validation X-rays found — falling back to TRAINING xrays for the '
              'visual demo (these have known disease labels, fine for a pipeline check, '
              'just not a blind test).')
        TEST_IMG_DIR = _fallback_train_img_dir
    else:
        raise FileNotFoundError('❌ No test images found anywhere. Check DENTEX_ROOT.')

OUTPUT_DIR = '/kaggle/working/stage3_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONFIG3 = {
    'conf_threshold': 0.6,   # Stage 1 detection confidence
    'nms_iou':        0.3,   # Stage 1 NMS IoU
    'padding':          20,  # Stage 1 crop-to-teeth-region padding
    'num_test_images':   5,
    'seed':              7,
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(CONFIG3['seed'])

print()
print(f'[CELL 1] CONFIG3 = {json.dumps(CONFIG3, indent=2)}')
print(f'[CELL 1] ✓ Device: {DEVICE}')
print(f'[CELL 1] ✓ Output dir: {OUTPUT_DIR}')
print('[CELL 1] ✓ Setup complete.')


In [ ]:
# ================== CELL 2: LOAD STAGE 1 MODEL ==================
def get_stage1_model(num_classes=33):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    in_feat_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_feat_mask, 256, num_classes)
    return model

print('[CELL 2] Building Stage 1 Mask R-CNN architecture (33 classes)...')
stage1_model = get_stage1_model(num_classes=33)

ckpt1 = torch.load(STAGE1_WEIGHTS, map_location=DEVICE)
if isinstance(ckpt1, dict) and 'model_state_dict' in ckpt1:
    stage1_model.load_state_dict(ckpt1['model_state_dict'], strict=False)
    print(f'  Loaded from model_state_dict. val_loss at save time: {ckpt1.get("val_loss", "N/A")}')
else:
    stage1_model.load_state_dict(ckpt1, strict=False)
    print('  Loaded from raw state_dict (no metadata wrapper found).')

stage1_model.to(DEVICE).eval()
n_params1 = sum(p.numel() for p in stage1_model.parameters())
print(f'  ✓ Stage 1 loaded. Total params: {n_params1:,}')

# ── GPU smoke test (catches kernel-image / accelerator mismatches early) ───
print('[CELL 2] Running GPU smoke test...')
try:
    _dummy = torch.rand(3, 64, 64)
    with torch.no_grad():
        _ = stage1_model([_dummy.to(DEVICE)])
    print(f'  ✓ Smoke test passed on {DEVICE}.')
except RuntimeError as e:
    print(f'  ✗ Smoke test FAILED on {DEVICE}: {e}')
    if DEVICE.type == 'cuda':
        print('  ACTION: Settings -> Accelerator -> switch to "GPU T4 x2", then re-run from Cell 1.')
    raise


In [ ]:
# ================== CELL 3: LOAD STAGE 2 MODEL ==================
def get_disease_classifier(num_classes, dropout=0.5):
    backbone = torchvision.models.resnet34(weights=None)
    in_features = backbone.fc.in_features
    backbone.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return backbone

print('[CELL 3] Loading Stage 2 checkpoint...')
ckpt2 = torch.load(STAGE2_WEIGHTS, map_location=DEVICE)

if not isinstance(ckpt2, dict) or 'classes' not in ckpt2:
    raise ValueError(
        '❌ Stage 2 checkpoint is missing expected metadata (classes/crop_pad/img_size/...). '
        'Make sure you attached stage2_disease_best.pth, not stage2_disease_final.pth '
        '(the "final" file is a raw state_dict only, saved without this metadata).'
    )

STAGE2_CLASSES      = {int(k): v for k, v in ckpt2['classes'].items()}
STAGE2_CLASS_COLORS = {v: tuple(ckpt2['class_colors'][v]) for v in STAGE2_CLASSES.values()}
STAGE2_CROP_PAD      = ckpt2['crop_pad']
STAGE2_IMG_SIZE      = ckpt2['img_size']
STAGE2_NORM_MEAN     = ckpt2['norm_mean']
STAGE2_NORM_STD      = ckpt2['norm_std']
NUM_DISEASE_CLASSES  = len(STAGE2_CLASSES)

print(f'  Trained at epoch {ckpt2.get("epoch", "N/A")}  (val_macro_f1={ckpt2.get("val_macro_f1", "N/A")})')
print(f'  Classes      : {STAGE2_CLASSES}')
print(f'  Class colors : {STAGE2_CLASS_COLORS}')
print(f'  crop_pad={STAGE2_CROP_PAD}  img_size={STAGE2_IMG_SIZE}')

stage2_model = get_disease_classifier(NUM_DISEASE_CLASSES)
stage2_model.load_state_dict(ckpt2['model_state_dict'])
stage2_model.to(DEVICE).eval()
n_params2 = sum(p.numel() for p in stage2_model.parameters())
print(f'  ✓ Stage 2 loaded. Total params: {n_params2:,}')

_stage2_tfm = T.Compose([
    T.Resize((STAGE2_IMG_SIZE, STAGE2_IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=STAGE2_NORM_MEAN, std=STAGE2_NORM_STD),
])
print('[CELL 3] ✓ Stage 2 ready.')


In [ ]:
# ================== CELL 4: STAGE 1 INFERENCE + QUADRANT/NUMBERING HELPERS ==================
# Ported verbatim from stage1_segmentation/teeth_segmentation.ipynb (Cell 6) so
# this pipeline produces IDENTICAL Stage 1 visualizations. Stage 1 itself is
# untouched — these are just the same post-processing functions, reused here.

def apply_nms(predictions, iou_threshold=CONFIG3['nms_iou']):
    if len(predictions['boxes']) == 0:
        return predictions
    keep_indices = nms(boxes=predictions['boxes'], scores=predictions['scores'],
                        iou_threshold=iou_threshold)
    return {k: v[keep_indices] for k, v in predictions.items()}


def run_inference(model, image_tensor, device, confidence_threshold=CONFIG3['conf_threshold']):
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.to(device).unsqueeze(0)
        raw_pred = model(image_tensor)[0]
    keep = raw_pred['scores'] >= confidence_threshold
    filtered = {k: v[keep] for k, v in raw_pred.items()}
    filtered = apply_nms(filtered)
    return {
        'boxes':  filtered['boxes'].cpu().numpy(),
        'labels': filtered['labels'].cpu().numpy(),
        'masks':  filtered['masks'].cpu().numpy(),
        'scores': filtered['scores'].cpu().numpy(),
    }


def crop_to_teeth_region(image, predictions, padding=CONFIG3['padding']):
    boxes = predictions['boxes']
    if len(boxes) == 0:
        return image, None
    x_min = int(boxes[:, 0].min()); y_min = int(boxes[:, 1].min())
    x_max = int(boxes[:, 2].max()); y_max = int(boxes[:, 3].max())
    x_min = max(0, x_min - padding); y_min = max(0, y_min - padding)
    x_max = min(image.shape[1], x_max + padding); y_max = min(image.shape[0], y_max + padding)
    crop_box = (x_min, y_min, x_max, y_max)
    return image[y_min:y_max, x_min:x_max], crop_box


def color_teeth(image, predictions, crop_box=None):
    if image.ndim == 2:
        colored_image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
        colored_image = image.copy()
    vibrant_colors = [
        [255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 255, 0], [255, 0, 255], [0, 255, 255],
        [255, 128, 0], [128, 0, 255], [255, 0, 128], [0, 255, 128], [128, 255, 0], [0, 128, 255],
    ]
    for i, (mask, label, box, score) in enumerate(zip(
        predictions['masks'], predictions['labels'], predictions['boxes'], predictions['scores'],
    )):
        mask_binary = (mask[0] > 0.5).astype(np.uint8)
        if crop_box is not None:
            x1, y1, x2, y2 = crop_box
            mask_cropped = mask_binary[y1:y2, x1:x2]
        else:
            mask_cropped = mask_binary
        if mask_cropped.shape != colored_image.shape[:2]:
            continue
        color = vibrant_colors[i % len(vibrant_colors)]
        overlay = colored_image.copy()
        for c in range(3):
            overlay[:, :, c] = np.where(mask_cropped == 1, color[c], overlay[:, :, c])
        colored_image = cv2.addWeighted(colored_image, 0.3, overlay, 0.7, 0)
        contours, _ = cv2.findContours(mask_cropped, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(colored_image, contours, -1, color, 3)
    return colored_image


def split_into_quadrants(predictions, image_shape, crop_box=None):
    if crop_box:
        x1, y1, x2, y2 = crop_box
        center_x = (x1 + x2) // 2; center_y = (y1 + y2) // 2
    else:
        height, width = image_shape[:2]
        center_x = width // 2; center_y = height // 2
    quadrants = {'UL': [], 'UR': [], 'LL': [], 'LR': []}
    for i, (box, label, mask, score) in enumerate(zip(
        predictions['boxes'], predictions['labels'], predictions['masks'], predictions['scores'],
    )):
        cx = int((box[0] + box[2]) / 2); cy = int((box[1] + box[3]) / 2)
        if   cx < center_x and cy < center_y: q = 'UL'
        elif cx >= center_x and cy < center_y: q = 'UR'
        elif cx < center_x and cy >= center_y: q = 'LL'
        else:                                   q = 'LR'
        quadrants[q].append({
            'index': i, 'label': int(label), 'box': box,
            'centroid': (cx, cy), 'mask': mask, 'score': float(score),
        })
    return quadrants, (center_x, center_y)


def number_teeth_in_quadrants(quadrants, center):
    numbered_quadrants = {}
    center_x, _ = center
    for q_name, teeth in quadrants.items():
        if not teeth:
            numbered_quadrants[q_name] = []
            continue
        if q_name in ['UL', 'LL']:
            sorted_teeth = sorted(teeth, key=lambda t: -t['centroid'][0])
        else:
            sorted_teeth = sorted(teeth, key=lambda t: t['centroid'][0])
        for num, tooth in enumerate(sorted_teeth, start=1):
            tooth['number'] = num
        numbered_quadrants[q_name] = sorted_teeth
    for q_name, teeth in numbered_quadrants.items():
        if len(teeth) > 8:
            teeth_sorted = sorted(teeth, key=lambda t: t['score'], reverse=True)
            numbered_quadrants[q_name] = teeth_sorted[:8]
    return numbered_quadrants

print('[CELL 4] ✓ Stage 1 inference + quadrant/numbering helpers ready '
      '(ported verbatim from stage1_segmentation/teeth_segmentation.ipynb).')


In [ ]:
# ================== CELL 5: PER-TOOTH STAGE 2 CLASSIFICATION ==================
def classify_tooth(image_np, box, pad=STAGE2_CROP_PAD):
    '''Crop one tooth from the ORIGINAL (uncropped) image and classify it with Stage 2.'''
    ih, iw = image_np.shape[:2]
    x1, y1, x2, y2 = box
    cx1 = max(0, int(x1) - pad); cy1 = max(0, int(y1) - pad)
    cx2 = min(iw, int(x2) + pad); cy2 = min(ih, int(y2) + pad)
    if cx2 <= cx1 or cy2 <= cy1:
        return {'label': 0, 'label_name': STAGE2_CLASSES[0], 'confidence': 0.0}
    crop = Image.fromarray(image_np[cy1:cy2, cx1:cx2]).convert('RGB')
    x = _stage2_tfm(crop).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = stage2_model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred = int(probs.argmax())
    return {
        'label': pred,
        'label_name': STAGE2_CLASSES[pred],
        'confidence': float(probs[pred]),
        'probs': {STAGE2_CLASSES[i]: float(probs[i]) for i in range(NUM_DISEASE_CLASSES)},
    }


def classify_all_teeth(image_np, numbered_quadrants):
    '''Run Stage 2 on every tooth in numbered_quadrants. Mutates tooth dicts in place
    (adds 'disease_label', 'disease_name', 'disease_conf') and returns class tallies.'''
    tallies = {name: 0 for name in STAGE2_CLASSES.values()}
    n_done = 0
    for q_name, teeth in numbered_quadrants.items():
        for tooth in teeth:
            result = classify_tooth(image_np, tooth['box'])
            tooth['disease_label'] = result['label']
            tooth['disease_name']  = result['label_name']
            tooth['disease_conf']  = result['confidence']
            tallies[result['label_name']] += 1
            n_done += 1
    return tallies, n_done

print('[CELL 5] ✓ classify_tooth() / classify_all_teeth() ready '
      f'(crop_pad={STAGE2_CROP_PAD}, img_size={STAGE2_IMG_SIZE}).')


In [ ]:
# ================== CELL 6: FULL PIPELINE VISUALIZATION + CLINICAL REPORT ==================
def visualize_full_pipeline(image_path, save_path=None):
    print(f'  Loading: {os.path.basename(image_path)}')
    pil_img = Image.open(image_path).convert('RGB')
    image_np = np.array(pil_img)
    image_tensor = TF.to_tensor(pil_img)

    # ── Stage 1: detect, crop, color-by-instance, quadrant + number ────────
    predictions = run_inference(stage1_model, image_tensor, DEVICE)
    print(f'  Stage 1 detected {len(predictions["labels"])} teeth')

    cropped_img, crop_box = crop_to_teeth_region(image_np.copy(), predictions)
    colored_img           = color_teeth(cropped_img.copy(), predictions, crop_box)
    quadrants, center     = split_into_quadrants(predictions, image_np.shape, crop_box)
    numbered_quadrants    = number_teeth_in_quadrants(quadrants, center)

    # ── Stage 2: classify every numbered tooth ──────────────────────────────
    tallies, n_classified = classify_all_teeth(image_np, numbered_quadrants)
    print(f'  Stage 2 classified {n_classified} teeth: {tallies}')

    # ── Panel 5: disease-colored visualization (same crop framing as Stage 1) ──
    disease_img = cropped_img.copy()
    if disease_img.ndim == 2:
        disease_img = cv2.cvtColor(disease_img, cv2.COLOR_GRAY2RGB)
    cx0, cy0 = (crop_box[0], crop_box[1]) if crop_box else (0, 0)

    for q_name, teeth in numbered_quadrants.items():
        for tooth in teeth:
            mask_binary = (tooth['mask'][0] > 0.5).astype(np.uint8)
            if crop_box is not None:
                x1, y1, x2, y2 = crop_box
                mask_cropped = mask_binary[y1:y2, x1:x2]
            else:
                mask_cropped = mask_binary
            if mask_cropped.shape != disease_img.shape[:2]:
                continue
            color = STAGE2_CLASS_COLORS.get(tooth['disease_name'], (128, 128, 255))
            overlay = disease_img.copy()
            for c in range(3):
                overlay[:, :, c] = np.where(mask_cropped == 1, color[c], overlay[:, :, c])
            disease_img = cv2.addWeighted(disease_img, 0.35, overlay, 0.65, 0)
            contours, _ = cv2.findContours(mask_cropped, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(disease_img, contours, -1, color, 3)

    h, w = disease_img.shape[:2]
    for q_name, teeth in numbered_quadrants.items():
        for tooth in teeth:
            cx = tooth['centroid'][0] - cx0
            cy = tooth['centroid'][1] - cy0
            if not (0 <= cx < w and 0 <= cy < h):
                continue
            label_text = f"{q_name}{tooth['number']}"
            cv2.putText(disease_img, label_text, (cx - 28, cy - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4)
            cv2.putText(disease_img, label_text, (cx - 28, cy - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            status = tooth['disease_name'] if tooth['disease_name'] != 'Healthy' else 'OK'
            cv2.putText(disease_img, status, (cx - 28, cy + 14),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 3)
            cv2.putText(disease_img, status, (cx - 28, cy + 14),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1)

    # ── Figure: Stage 1's 4 panels (unchanged) + disease panel + legend ────
    fig = plt.figure(figsize=(22, 14))
    gs = fig.add_gridspec(2, 3, hspace=0.2, wspace=0.15)

    ax1 = fig.add_subplot(gs[0, 0]); ax1.imshow(image_np, cmap='gray')
    ax1.set_title('Original Panoramic X-ray', fontsize=15, fontweight='bold'); ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 1]); ax2.imshow(cropped_img, cmap='gray')
    ax2.set_title('Stage 1: Cropped to Teeth Region', fontsize=15, fontweight='bold'); ax2.axis('off')

    ax3 = fig.add_subplot(gs[0, 2]); ax3.imshow(colored_img)
    ax3.set_title('Stage 1: Each Tooth Colored by Instance', fontsize=15, fontweight='bold'); ax3.axis('off')

    quad_img = colored_img.copy()
    qh, qw = quad_img.shape[:2]
    cxq, cyq = center[0] - cx0, center[1] - cy0
    cv2.line(quad_img, (cxq, 0), (cxq, qh), (255, 255, 0), 4)
    cv2.line(quad_img, (0, cyq), (qw, cyq), (255, 255, 0), 4)
    for q_name, teeth in numbered_quadrants.items():
        for tooth in teeth:
            cx, cy = tooth['centroid'][0] - cx0, tooth['centroid'][1] - cy0
            if not (0 <= cx < qw and 0 <= cy < qh):
                continue
            num = tooth['number']
            cv2.putText(quad_img, str(num), (cx - 20, cy + 16), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 0, 0), 6)
            cv2.putText(quad_img, str(num), (cx - 20, cy + 16), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 255, 255), 3)
    ax4 = fig.add_subplot(gs[1, 0]); ax4.imshow(quad_img)
    ax4.set_title('Stage 1: Quadrants + Numbering (1-8)', fontsize=15, fontweight='bold'); ax4.axis('off')

    ax5 = fig.add_subplot(gs[1, 1]); ax5.imshow(disease_img)
    ax5.set_title('Stage 2: Disease Status per Tooth', fontsize=15, fontweight='bold'); ax5.axis('off')

    ax6 = fig.add_subplot(gs[1, 2]); ax6.axis('off')
    ax6.set_title('Legend & Summary', fontsize=15, fontweight='bold')
    y = 0.92
    for name, color in STAGE2_CLASS_COLORS.items():
        ax6.add_patch(plt.Rectangle((0.05, y - 0.03), 0.08, 0.06,
                                     color=tuple(c / 255 for c in color), transform=ax6.transAxes))
        ax6.text(0.17, y, f'{name}  (n={tallies.get(name, 0)})', transform=ax6.transAxes,
                 fontsize=12, va='center')
        y -= 0.10
    ax6.text(0.05, y - 0.05, f'Total teeth classified: {n_classified}', transform=ax6.transAxes, fontsize=11)

    plt.suptitle(os.path.basename(image_path), fontsize=13, y=0.99)
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f'  ✓ Saved figure -> {save_path}')
    plt.show()

    # ── Clinical report (text) ──────────────────────────────────────────────
    print()
    print('='*70)
    print('CLINICAL REPORT')
    print('='*70)
    for q_name in ['UL', 'UR', 'LL', 'LR']:
        teeth = numbered_quadrants[q_name]
        print(f'\n{q_name} Quadrant: {len(teeth)} teeth')
        for tooth in sorted(teeth, key=lambda t: t['number']):
            flag = '' if tooth['disease_name'] == 'Healthy' else '  ⚠ FLAGGED'
            print(f"  Tooth {q_name}-{tooth['number']:<2d} | status: {tooth['disease_name']:<18s} "
                  f"| confidence: {tooth['disease_conf']*100:5.1f}%{flag}")

    return {
        'numbered_quadrants': numbered_quadrants,
        'tallies': tallies,
        'n_teeth': len(predictions['labels']),
        'n_classified': n_classified,
    }

print('[CELL 6] ✓ visualize_full_pipeline() ready.')


In [ ]:
# ================== CELL 7: RUN ON SAMPLE TEST IMAGES ==================
all_test_files = sorted([f for f in os.listdir(TEST_IMG_DIR)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
print(f'[CELL 7] Found {len(all_test_files)} candidate test images in {TEST_IMG_DIR}')

random.seed(CONFIG3['seed'])
sample_files = random.sample(all_test_files, min(CONFIG3['num_test_images'], len(all_test_files)))
print(f'  Running pipeline on {len(sample_files)} sample images: {sample_files}')

all_results = []
for i, fname in enumerate(sample_files):
    print()
    print('='*70)
    print(f'SAMPLE {i+1}/{len(sample_files)}: {fname}')
    print('='*70)
    fpath = os.path.join(TEST_IMG_DIR, fname)
    save_path = os.path.join(OUTPUT_DIR, f'pipeline_{os.path.splitext(fname)[0]}.png')
    result = visualize_full_pipeline(fpath, save_path=save_path)
    result['file'] = fname
    all_results.append(result)

print()
print('[CELL 7] ✓ Pipeline run complete on all sample images.')


In [ ]:
# ================== CELL 8: SAVE PIPELINE MANIFEST + SAMPLE OUTPUTS ==================
# Stage 3 doesn't train a new model — it's an inference pipeline combining the two
# already-trained models. What we "save" here is a manifest describing exactly which
# checkpoints + settings produced these results, so the pipeline is reproducible.

manifest = {
    'stage1_weights_path': STAGE1_WEIGHTS,
    'stage2_weights_path': STAGE2_WEIGHTS,
    'stage2_classes': STAGE2_CLASSES,
    'stage2_class_colors': {k: list(v) for k, v in STAGE2_CLASS_COLORS.items()},
    'stage2_val_macro_f1_at_save': ckpt2.get('val_macro_f1', None),
    'pipeline_config': CONFIG3,
    'test_image_dir': TEST_IMG_DIR,
    'samples_run': [r['file'] for r in all_results],
    'per_sample_tallies': {r['file']: r['tallies'] for r in all_results},
}

manifest_path = os.path.join(OUTPUT_DIR, 'stage3_pipeline_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('[CELL 8] ✓ Saved pipeline manifest ->', manifest_path)
print()
print('[CELL 8] Files in', OUTPUT_DIR, ':')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'    {fname:<45s} {size_kb:>10.1f} KB')

print()
print('[CELL 8] ✓ STAGE 3 PIPELINE COMPLETE.')
print('  Aggregate disease tallies across all sample images:')
agg = {}
for r in all_results:
    for k, v in r['tallies'].items():
        agg[k] = agg.get(k, 0) + v
for k, v in agg.items():
    print(f'    {k:<20s}: {v}')
